# Exercice technique Verbus

## Objectif de l'analyse

## Chargement des données

In [10]:
import pandas as pd
import json
import time

df_history = pd.read_csv('data/bornes_history.csv')
df_history["timestamp"] = pd.to_datetime(df_history["timestamp"], format='ISO8601', utc=True)
df_planning = pd.read_csv('data/planning.csv', parse_dates =["arrival_time", "departure_time"])
df_sessions = pd.read_csv('data/sessions.csv', parse_dates=["start_time", "end_time"])

with open('data/bornes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df_bornes = pd.DataFrame(data)
df_bornes["last_update"] = pd.to_datetime(df_bornes["last_update"], format='ISO8601', utc=True)

## Exploration initiale des données

In [77]:
df_bornes.info()
df_bornes.head()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   station_id    6 non-null      str                
 1   connector_id  6 non-null      int64              
 2   status        6 non-null      str                
 3   power_kw      6 non-null      int64              
 4   last_update   6 non-null      datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(2), str(2)
memory usage: 372.0 bytes


,station_id,connector_id,status,power_kw,last_update
0,STAT-01,1,Available,11,2025-04-13 19:00:00+00:00
1,STAT-01,2,Available,11,2025-04-08 04:00:00+00:00
2,STAT-02,1,Available,11,2025-04-07 06:00:00+00:00
3,STAT-02,2,Faulted,11,2025-04-09 08:00:00+00:00
4,STAT-03,1,Unavailable,11,2025-04-09 22:00:00+00:00


In [51]:
df_bornes.info()
df_bornes.head()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   station_id    6 non-null      str                
 1   connector_id  6 non-null      int64              
 2   status        6 non-null      str                
 3   power_kw      6 non-null      int64              
 4   last_update   6 non-null      datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(2), str(2)
memory usage: 372.0 bytes


,station_id,connector_id,status,power_kw,last_update
0,STAT-01,1,Available,11,2025-04-13 19:00:00+00:00
1,STAT-01,2,Available,11,2025-04-08 04:00:00+00:00
2,STAT-02,1,Available,11,2025-04-07 06:00:00+00:00
3,STAT-02,2,Faulted,11,2025-04-09 08:00:00+00:00
4,STAT-03,1,Unavailable,11,2025-04-09 22:00:00+00:00


In [52]:
df_planning.info()

<class 'pandas.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   vehicle_id           172 non-null    str                
 1   vehicle_type         172 non-null    str                
 2   arrival_time         172 non-null    datetime64[us, UTC]
 3   departure_time       172 non-null    datetime64[us, UTC]
 4   service_km           172 non-null    float64            
 5   service_hours        172 non-null    float64            
 6   required_charge_kwh  172 non-null    float64            
dtypes: datetime64[us, UTC](2), float64(3), str(2)
memory usage: 9.5 KB


In [53]:
df_planning.info()

<class 'pandas.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   vehicle_id           172 non-null    str                
 1   vehicle_type         172 non-null    str                
 2   arrival_time         172 non-null    datetime64[us, UTC]
 3   departure_time       172 non-null    datetime64[us, UTC]
 4   service_km           172 non-null    float64            
 5   service_hours        172 non-null    float64            
 6   required_charge_kwh  172 non-null    float64            
dtypes: datetime64[us, UTC](2), float64(3), str(2)
memory usage: 9.5 KB


In [54]:
df_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   station_id    111 non-null    str                
 1   connector_id  111 non-null    int64              
 2   timestamp     111 non-null    datetime64[us, UTC]
 3   status        111 non-null    str                
dtypes: datetime64[us, UTC](1), int64(1), str(2)
memory usage: 3.6 KB


In [55]:
df_sessions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   session_id          52 non-null     str                
 1   station_id          52 non-null     str                
 2   connector_id        52 non-null     int64              
 3   vehicle_id          52 non-null     str                
 4   start_time          52 non-null     datetime64[us, UTC]
 5   end_time            51 non-null     datetime64[us, UTC]
 6   energy_kwh          52 non-null     float64            
 7   effective_power_kw  52 non-null     int64              
dtypes: datetime64[us, UTC](2), float64(1), int64(2), str(3)
memory usage: 3.4 KB


## Compréhension des fichiers et des liens entre eux

bornes.json décrit l’infrastructure de recharge “à l’instant T” : pour chaque station et connecteur, on connaît le statut courant (Available, Charging, Faulted, Unavailable), la puissance nominale (11 kW) et la date de dernière mise à jour.

sessions.csv retrace, sur la période étudiée, les sessions de recharge réalisées : chaque ligne correspond à une session sur un connecteur donné, pour un véhicule donné, avec les horodatages de début/fin, l’énergie délivrée et la puissance effective de charge.

planning.csv décrit le planning d’exploitation de la flotte : pour chaque véhicule et chaque sortie, on dispose des horaires d’arrivée/départ au dépôt, des kilomètres parcourus, de la durée de service et de l’énergie estimée nécessaire pour le prochain service.

bornes_history.csv détaille l’historique des changements d’état des bornes : pour chaque station et connecteur, on suit dans le temps les passages entre Available, Charging, Faulted et Unavailable.

Ensemble, ces quatre fichiers permettent de relier le besoin opérationnel (planning des véhicules et énergie requise) à l’usage réel des bornes (sessions de recharge) et à la disponibilité effective de l’infrastructure dans le temps (historique des statuts).

## Vérifications de cohérence

In [18]:
print("=== Vérifications de cohérence - bornes.json ===")

print("Nombre de connecteurs :", len(df_bornes))
print("Nombre de stations :", df_bornes["station_id"].nunique())

print("\nStatuts observés :")
print(df_bornes["status"].value_counts())

print("\nPuissances nominales distinctes (kW) :")
print(df_bornes["power_kw"].unique())

=== Vérifications de cohérence - bornes.json ===
Nombre de connecteurs : 6
Nombre de stations : 3

Statuts observés :
status
Available      4
Faulted        1
Unavailable    1
Name: count, dtype: int64

Puissances nominales distinctes (kW) :
[11]


In [ ]:
print("=== Vérifications de cohérence - sessions.csv ===")

print("Nombre de sessions :", len(df_sessions))
print("Nombre de véhicules distincts :", df_sessions["vehicle_id"].nunique())

print("\nSessions incomplètes :")
print(df_sessions["end_time"].isna().sum())

print("\nStatistiques énergie (kWh) :")
print(df_sessions["energy_kwh"].describe())

=== Vérifications de cohérence – sessions.csv ===
Nombre de sessions : 52
Nombre de véhicules distincts : 18

Sessions incomplètes :
1

Statistiques énergie (kWh) :
count    52.000000
mean     27.397115
std      13.988860
min       0.000000
25%      19.540000
50%      24.190000
75%      35.625000
max      62.170000
Name: energy_kwh, dtype: float64


In [ ]:
print("=== Vérifications de cohérence - planning.csv ===")

print("Nombre de lignes de planning :", len(df_planning))
print("Nombre de véhicules distincts :", df_planning["vehicle_id"].nunique())

print("\nRépartition des types de véhicules :")
print(df_planning["vehicle_type"].value_counts())

print("\nStatistiques service_km :")
print(df_planning["service_km"].describe())

print("\nStatistiques service_hours :")
print(df_planning["service_hours"].describe())

=== Vérifications de cohérence – planning.csv ===
Nombre de lignes de planning : 172
Nombre de véhicules distincts : 20

Répartition des types de véhicules :
vehicle_type
Renault Zoé      70
Citroën Jumpy    53
Kia EV3          49
Name: count, dtype: int64

Statistiques service_km :
count    172.000000
mean     148.786628
std       38.420450
min       81.300000
25%      121.250000
50%      149.950000
75%      172.900000
max      248.700000
Name: service_km, dtype: float64

Statistiques service_hours :
count    172.000000
mean       6.127326
std        1.636680
min        3.100000
25%        4.975000
50%        6.200000
75%        7.500000
max        9.000000
Name: service_hours, dtype: float64


In [9]:
print("=== Vérifications de cohérence - bornes_history.csv ===")

print("Nombre total d'événements :", len(df_history))

print("\nStations couvertes :")
print(df_history["station_id"].unique())

print("Nombre de connecteurs distincts :",
      df_history[["station_id", "connector_id"]].drop_duplicates().shape[0])

print("\nStatuts observés :")
print(df_history["status"].value_counts())

print("\nPériode couverte :")
print("Min :", df_history["timestamp"].min())
print("Max :", df_history["timestamp"].max())

=== Vérifications de cohérence - bornes_history.csv ===
Nombre total d'événements : 111

Stations couvertes :
<StringArray>
['STAT-01', 'STAT-02', 'STAT-03']
Length: 3, dtype: str
Nombre de connecteurs distincts : 6

Statuts observés :
status
Available      57
Charging       52
Faulted         1
Unavailable     1
Name: count, dtype: int64

Période couverte :
Min : 2025-04-07 00:00:00+00:00
Max : 2025-04-11 20:00:00+00:00


Pour l’analyse de l’utilisation des bornes, je me limite aux sessions “valides”,
c’est‑à‑dire celles quiont une fin (end_time renseigné) et au moins 1 kWh délivré ; les
sessions incomplètes ou quasinulles sont analysées séparément comme anomalies.
Sur cette base, les 6 connecteurs délivrent au total 1 423,01 kWh, avec entre 3 et 11
sessions valides par connecteur.
Les connecteurs les plus sollicités sont STAT‑01‑2 (11 sessions, 272,98 kWh) et
STAT‑03‑1 (10 sessions, 223,42 kWh), tandis que STAT‑02‑2 n’est utilisé que 3 fois (94,66
kWh).
L’énergie moyenne par session valide varie d’environ 22 kWh(STAT‑03‑1) à 37 kWh
(STAT‑03‑2), ce qui correspond à des compléments de charge significatifs plutôt qu’à
des recharges complètes.

## KPI 1 - Utilisation des bornes

In [ ]:
print("=== KPI 1 – Utilisation des bornes (sessions valides) ===")

sessions_valides = df_sessions[
    (df_sessions["end_time"].notna()) &
    (df_sessions["energy_kwh"] >= 1)
]

kpi_bornes = (
    sessions_valides
    .groupby(["station_id", "connector_id"])
    .agg(
        nb_sessions=("session_id", "size"),
        energie_totale_kwh=("energy_kwh", "sum"),
        energie_moyenne_kwh=("energy_kwh", "mean")
    )
    .reset_index()
    .sort_values(["station_id", "connector_id"])
)

display(kpi_bornes)

print(
    "\nÉnergie totale délivrée sur la période (sessions valides, kWh) :",
    sessions_valides["energy_kwh"].sum()
)

=== KPI 1 – Utilisation des bornes (sessions valides) ===


,station_id,connector_id,nb_sessions,energie_totale_kwh,energie_moyenne_kwh
0,STAT-01,1,8,257.67,32.208750
1,STAT-01,2,11,272.98,24.816364
2,STAT-02,1,9,314.84,34.982222
3,STAT-02,2,3,94.66,31.553333
4,STAT-03,1,10,223.42,22.342000
5,STAT-03,2,7,259.44,37.062857



Énergie totale délivrée sur la période (sessions valides, kWh) : 1423.01


Pour l’analyse de l’utilisation des bornes, je me limite aux sessions “valides”, c’est‑à‑dire celles qui ont une fin (end_time renseigné) et au moins 1 kWh délivré ; les sessions incomplètes ou quasi nulles sont analysées séparément comme anomalies.

Sur cette base, les 6 connecteurs délivrent au total 1 423,01 kWh, avec entre 3 et 11 sessions valides par connecteur.

Les connecteurs les plus sollicités sont STAT‑01‑2 (11 sessions, 272,98 kWh) et STAT‑03‑1 (10 sessions, 223,42 kWh), tandis que STAT‑02‑2 n’est utilisé que 3 fois (94,66 kWh).

L’énergie moyenne par session valide varie d’environ 22 kWh (STAT‑03‑1) à 37 kWh (STAT‑03‑2), ce qui correspond à des compléments de charge significatifs plutôt qu’à des recharges complètes.

## KPI 2 - Indisponibilité des bornes

In [ ]:
import pandas as pd

# On suppose que df_history existe déjà avec les colonnes :
# station_id, connector_id, timestamp, status (Available / Charging / Faulted / Unavailable)

# 1) Mise au bon format
print("Valeurs possibles de status :", df_history["status"].unique())
df_history["timestamp"] = pd.to_datetime(df_history["timestamp"])

# 2) On trie par borne, prise et temps
df_history = df_history.sort_values(
    ["station_id", "connector_id", "timestamp"]
)

# 3) Fin de chaque intervalle d'état = timestamp de la ligne suivante (par borne/prise)
df_history["next_timestamp"] = df_history.groupby(
    ["station_id", "connector_id"]
)["timestamp"].shift(-1)

# 4) On garde seulement les lignes où on connaît la durée (où on a un next_timestamp)
df_intervals = df_history.dropna(subset=["next_timestamp"]).copy()

# 5) Durée de chaque intervalle en heures
df_intervals["duration_h"] = (
    df_intervals["next_timestamp"] - df_intervals["timestamp"]
).dt.total_seconds() / 3600

# 6) États d'indisponibilité : Faulted ou Unavailable
df_intervals["status_clean"] = df_intervals["status"].str.strip().str.lower()
df_intervals["is_unavailable"] = df_intervals["status_clean"].isin(
    ["faulted", "unavailable"]
)

print("Nombre d'intervalles indisponibles :", df_intervals["is_unavailable"].sum())
print("Exemple d'intervalles indisponibles :")
print(df_intervals[df_intervals["is_unavailable"]].head())

# 7) Temps total et temps indisponible (toutes bornes confondues)
total_hours = df_intervals["duration_h"].sum()
unavailable_hours = df_intervals.loc[df_intervals["is_unavailable"], "duration_h"].sum()

taux_indispo = unavailable_hours / total_hours if total_hours > 0 else 0

print(f"Taux global d'indisponibilité des bornes : {taux_indispo:.1%}")

<StringArray>
['Available', 'Charging', 'Faulted', 'Unavailable']
Length: 4, dtype: str
0
Empty DataFrame
Columns: [station_id, connector_id, timestamp, status, next_timestamp, duration_h, status_clean, is_unavailable]
Index: []
Taux global d'indisponibilité des bornes : 0.0%


## KPI 3 - Recharge des véhicules et couverture des besoins

In [ ]:

besoin_par_vehicule = (
    df_planning
    .groupby("vehicle_id", as_index=False)["required_charge_kwh"]
    .sum()
)


recharge_par_vehicule = (
    df_sessions
    .groupby("vehicle_id", as_index=False)["energy_kwh"]
    .sum()
)


df_kpi3 = besoin_par_vehicule.merge(
    recharge_par_vehicule,
    on="vehicle_id",
    how="left"
)


df_kpi3["energy_kwh"] = df_kpi3["energy_kwh"].fillna(0)


df_kpi3["taux_couverture"] = (
    df_kpi3["energy_kwh"] / df_kpi3["required_charge_kwh"]
)


besoin_total = df_kpi3["required_charge_kwh"].sum()
recharge_totale = df_kpi3["energy_kwh"].sum()

taux_couverture_global = (
    recharge_totale / besoin_total if besoin_total > 0 else 0
)

print(df_kpi3.head())
print(f"Taux global de couverture des besoins de recharge : {taux_couverture_global:.1%}")

  vehicle_id  required_charge_kwh  energy_kwh  taux_couverture
0    VEH-001               196.86       56.60         0.287514
1    VEH-002               219.24       93.85         0.428070
2    VEH-003               232.98       58.24         0.249979
3    VEH-004               130.52      102.14         0.782562
4    VEH-005               225.62       53.01         0.234953
Taux global de couverture des besoins de recharge : 26.1%


Sur ce KPI, on voit que les véhicules sont globalement moins rechargés que ce qui était prévu dans le planning.

Par exemple, VEH‑001 ne reçoit qu’environ 57 kWh sur presque 197 kWh demandés, soit à peine 29% de ses besoins. VEH‑003 est dans le même cas, avec environ 25% de couverture seulement. VEH‑004 s’en sort mieux avec un peu plus de 78% de ses besoins couverts.

Au total, si on additionne tous les véhicules, la flotte ne reçoit qu’environ 26% de l’énergie qui était théoriquement nécessaire sur la période.

## KPI 4 - Créneaux de tension

In [ ]:
evenements_debut = df_sessions[["start_time"]].copy()
evenements_debut["delta"] = 1

evenements_fin = df_sessions[["end_time"]].copy()
evenements_fin["delta"] = -1

evenements_debut = evenements_debut.rename(columns={"start_time": "instant"})
evenements_fin = evenements_fin.rename(columns={"end_time": "instant"})

evenements = pd.concat([evenements_debut, evenements_fin], ignore_index=True)
evenements = evenements.sort_values(["instant", "delta"])

nb_actuel_en_charge = 0
lignes = []

for _, ligne in evenements.iterrows():
    nb_actuel_en_charge += ligne["delta"]
    lignes.append({"instant": ligne["instant"], "nb_en_charge": nb_actuel_en_charge})

chronologie = pd.DataFrame(lignes)
chronologie_5_plus = chronologie[chronologie["nb_en_charge"] >= 5]

,time,nb_en_charge
8,2025-04-07 14:10:16.927315+00:00,5
28,2025-04-08 13:25:07.637138+00:00,5
30,2025-04-08 13:35:59.162823+00:00,5
32,2025-04-08 14:52:40.124603+00:00,5
52,2025-04-09 13:02:17.997184+00:00,5
53,2025-04-09 13:41:20.475568+00:00,6
54,2025-04-09 14:32:08.702995+00:00,5
55,2025-04-09 14:40:16.891591+00:00,6
56,2025-04-09 14:59:42.273950+00:00,5
58,2025-04-09 15:33:29.162171+00:00,5


# KPI 5 – Créneaux de tension

Pour ce KPI, je calcule combien de véhicules sont en charge en même temps, en comptant +1 à chaque début de session et −1 à chaque fin. Dès que ce nombre atteint au moins 5 véhicules, je considère que l’on est dans un créneau de tension. 

## Visualisations

## Réponse à la question A - L'infrastructure est-elle suffisante ?

## Réponse à la question B - Données manquantes ou peu fiables

## Limites et pistes d'amélioration